<a href="https://www.kaggle.com/code/asivakumarnair/diabetic-retinopathy-imagenet?scriptVersionId=343261398" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [ ]:
# ===== FULL REBUILD, NEW SESSION: environment, data, reload 4 pooled models, get predictions =====

!pip install -q tensorflow==2.19.0

import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'

import random
import numpy as np
import pandas as pd
import gc
import tensorflow as tf

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print(f"Seed {SEED} set, TF {tf.__version__}, tf.keras module: {tf.keras.__name__}")

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.efficientnet import preprocess_input as eff_pre
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mob_pre
from tensorflow.keras.applications.resnet50 import preprocess_input as res_pre
from tensorflow.keras.models import load_model
from sklearn.model_selection import train_test_split

# ---------- CONFIG ----------
APTOS_CSV     = '/kaggle/input/competitions/aptos2019-blindness-detection/train.csv'
APTOS_IMG     = '/kaggle/input/competitions/aptos2019-blindness-detection/train_images'
EYEPACS_CSV   = '/kaggle/input/datasets/benjaminwarner/resized-2015-2019-blindness-detection-images/labels/trainLabels15.csv'
EYEPACS_IMG   = '/kaggle/input/datasets/benjaminwarner/resized-2015-2019-blindness-detection-images/resized train 15'
MESSIDOR_CSV  = '/kaggle/input/datasets/mariaherrerot/messidor2preprocess/messidor_data.csv'
MESSIDOR_IMG  = '/kaggle/input/datasets/mariaherrerot/messidor2preprocess/messidor-2/messidor-2/preprocess'

WEIGHTS_DIR   = '/kaggle/input/datasets/asivakumarnair/drbestmodels/'

GRADES      = ['0','1','2','3','4']
NUM_CLASSES = 5
IMG_SIZE    = 224
BATCH_SIZE  = 32
SUBSAMPLE_SEED = 42
EYEPACS_TARGET = 3662

W_CUSTOM = WEIGHTS_DIR + 'dr_best_custom_cnn.keras'
W_EFF    = WEIGHTS_DIR + 'dr_best_efficientnet.keras'
W_MOB    = WEIGHTS_DIR + 'dr_best_mobilenet.keras'
W_RES    = WEIGHTS_DIR + 'dr_best_resnet50.keras'

for name, path in [('custom',W_CUSTOM),('eff',W_EFF),('mob',W_MOB),('res',W_RES)]:
    print(f"{name}: {'FOUND' if os.path.exists(path) else 'MISSING'} -> {path}")

# ---------- DATA REBUILD ----------
aptos = pd.read_csv(APTOS_CSV)
aptos['grade']      = aptos['diagnosis'].astype(int).astype(str)
aptos['image_path'] = APTOS_IMG + '/' + aptos['id_code'].astype(str) + '.png'
aptos['source']     = 'aptos'
aptos['patient_id'] = None

eyepacs = pd.read_csv(EYEPACS_CSV)
eyepacs['grade']      = eyepacs['level'].astype(int).astype(str)
eyepacs['image_path'] = EYEPACS_IMG + '/' + eyepacs['image'].astype(str) + '.jpg'
eyepacs['source']     = 'eyepacs'
eyepacs['patient_id'] = eyepacs['image'].str.extract(r'^(\d+)_')

messidor = pd.read_csv(MESSIDOR_CSV)
messidor['grade']      = messidor['diagnosis'].astype(int).astype(str)
messidor['image_path'] = MESSIDOR_IMG + '/' + messidor['id_code'].astype(str)
messidor['source']     = 'messidor'
messidor['patient_id'] = None

def subsample_eyepacs(df, target_n=EYEPACS_TARGET, seed=SUBSAMPLE_SEED):
    pg = df.groupby('patient_id')['grade'].max().reset_index()
    frac = target_n / len(df)
    keep, _ = train_test_split(pg, train_size=frac, stratify=pg['grade'], random_state=seed)
    return df[df['patient_id'].isin(keep['patient_id'])].reset_index(drop=True)

eyepacs_s = subsample_eyepacs(eyepacs)

def safe_split(df, label_col, test_size, rs, tag=""):
    try:
        return train_test_split(df, test_size=test_size, stratify=df[label_col], random_state=rs)
    except ValueError as e:
        print(f"WARNING [{tag}]: stratified split failed, falling back to unstratified.")
        return train_test_split(df, test_size=test_size, random_state=rs)

def split_patient_level(df, rs=SEED, tag=""):
    pg = df.groupby('patient_id')['grade'].max().reset_index()
    p_tr, p_tmp = safe_split(pg, 'grade', 0.30, rs, tag=f"{tag} first")
    p_va, p_te  = safe_split(p_tmp, 'grade', 0.50, rs, tag=f"{tag} second")
    pick = lambda ids: df[df['patient_id'].isin(ids['patient_id'])]
    return pick(p_tr), pick(p_va), pick(p_te)

def split_image_level(df, rs=SEED, tag=""):
    tr, tmp = safe_split(df, 'grade', 0.30, rs, tag=f"{tag} first")
    va, te  = safe_split(tmp, 'grade', 0.50, rs, tag=f"{tag} second")
    return tr, va, te

def split_messidor_mixed(df, rs=SEED, tag="Messidor"):
    is_im = ~df['image_path'].str.contains(r'\d{8}_\d+_\d+_PP\.png$', regex=True)
    im_df = df[is_im].copy()
    im_df['im_num'] = im_df['image_path'].str.extract(r'IM(\d+)\.JPG$').astype(int)
    im_df = im_df.sort_values('im_num').reset_index(drop=True)
    im_df['patient_id'] = 'messidor_pair_' + (im_df.index // 2).astype(str)
    pg = im_df.groupby('patient_id')['grade'].max().reset_index()
    p_tr, p_tmp = safe_split(pg, 'grade', 0.30, rs, tag=f"{tag} IM first")
    p_va, p_te  = safe_split(p_tmp, 'grade', 0.50, rs, tag=f"{tag} IM second")
    pick = lambda ids: im_df[im_df['patient_id'].isin(ids['patient_id'])]
    im_tr, im_va, im_te = pick(p_tr), pick(p_va), pick(p_te)
    date_df = df[~is_im]
    d_tr, d_va, d_te = split_image_level(date_df, rs, tag=f"{tag} date-style")
    cat = lambda a, b: pd.concat([a.drop(columns=['im_num']), b], ignore_index=True)
    return cat(im_tr, d_tr), cat(im_va, d_va), cat(im_te, d_te)

a_tr, a_va, a_te = split_image_level(aptos, tag="APTOS")
e_tr, e_va, e_te = split_patient_level(eyepacs_s, tag="EyePACS")
m_tr, m_va, m_te = split_messidor_mixed(messidor)

test_df = pd.concat([a_te, e_te, m_te], ignore_index=True)
print(f"Test set: {len(test_df):,} rows")

def make_gens_test_only(preprocess_fn, te_df):
    idg = (ImageDataGenerator(rescale=1./255) if preprocess_fn is None
           else ImageDataGenerator(preprocessing_function=preprocess_fn))
    return idg.flow_from_dataframe(te_df, x_col='image_path', y_col='grade',
                                    target_size=(IMG_SIZE,IMG_SIZE), batch_size=BATCH_SIZE,
                                    class_mode='categorical', classes=GRADES, color_mode='rgb', shuffle=False)

# ---------- RELOAD ALL FOUR, GET PREDICTIONS, SAVE IMMEDIATELY ----------
MODEL_PATHS = {
    'custom_cnn':     (W_CUSTOM, None),
    'mobilenetv2':    (W_MOB,    mob_pre),
    'efficientnetb0': (W_EFF,    eff_pre),
    'resnet50':       (W_RES,    res_pre),
}

y_true_ref = None
for name, (path, prep_fn) in MODEL_PATHS.items():
    print(f"\nLoading {name}...")
    m = load_model(path)
    te_gen = make_gens_test_only(prep_fn, test_df)
    y_true = te_gen.classes
    y_prob = m.predict(te_gen, verbose=0)
    y_pred = np.argmax(y_prob, axis=1)

    if y_true_ref is None:
        y_true_ref = y_true
    else:
        assert np.array_equal(y_true_ref, y_true), f"{name} test order does not match, stop and investigate"

    np.savez(f'/kaggle/working/preds_{name}.npz', y_true=y_true, y_pred=y_pred, y_prob=y_prob)
    print(f"  saved preds_{name}.npz, accuracy check: {(y_pred==y_true).mean():.4f}")

    del m
    gc.collect()
    tf.keras.backend.clear_session()

print("\nAll four models scored, predictions saved. y_true order confirmed identical across all four.")
print("Compare accuracy above against locked Stage 8 numbers: custom 0.622, eff 0.649, mob 0.671, res 0.679")

In [ ]:
# ===== STAGE 9: McNEMAR + DeLONG (plain and correlated), DR, 4 models, 5 classes =====
import numpy as np
import pandas as pd
from itertools import combinations
from scipy import stats
from statsmodels.stats.contingency_tables import mcnemar
from sklearn.metrics import roc_auc_score

MODEL_NAMES = ['custom_cnn', 'mobilenetv2', 'efficientnetb0', 'resnet50']
DISPLAY_NAMES = {'custom_cnn':'Custom CNN','mobilenetv2':'MobileNetV2',
                  'efficientnetb0':'EfficientNetB0','resnet50':'ResNet50'}
NUM_CLASSES = 5

preds = {}
for name in MODEL_NAMES:
    d = np.load(f'/kaggle/working/preds_{name}.npz')
    preds[name] = dict(y_true=d['y_true'], y_pred=d['y_pred'], y_prob=d['y_prob'])

base_yt = preds[MODEL_NAMES[0]]['y_true']
for name in MODEL_NAMES[1:]:
    assert np.array_equal(base_yt, preds[name]['y_true']), f"y_true mismatch for {name}"
print("y_true alignment check: PASS")
y_true = base_yt

pairs = list(combinations(MODEL_NAMES, 2))

# ---------- 9a: McNemar ----------
correct = {name: (preds[name]['y_pred'] == y_true) for name in MODEL_NAMES}
rows = []
for a, b in pairs:
    ca, cb = correct[a], correct[b]
    n10 = int(np.sum(ca & ~cb)); n01 = int(np.sum(~ca & cb))
    n11 = int(np.sum(ca & cb)); n00 = int(np.sum(~ca & ~cb))
    table = [[n11, n10], [n01, n00]]
    result = mcnemar(table, exact=(n10 + n01) < 25, correction=True)
    rows.append({'model_a': DISPLAY_NAMES[a], 'model_b': DISPLAY_NAMES[b],
                  'a_only_correct': n10, 'b_only_correct': n01,
                  'statistic': result.statistic, 'p_raw': result.pvalue})
mcnemar_df = pd.DataFrame(rows).sort_values('p_raw').reset_index(drop=True)
m = len(mcnemar_df)
mcnemar_df['p_holm'] = (mcnemar_df['p_raw'] * (m - mcnemar_df.index)).clip(upper=1.0).cummax()
mcnemar_df['significant_holm'] = mcnemar_df['p_holm'] < 0.05
mcnemar_df.to_csv('/kaggle/working/dr_stage9_mcnemar.csv', index=False)
print("\n===== McNemar =====")
print(mcnemar_df.to_string(index=False))

# ---------- 9b: DeLong, plain ----------
def delong_roc_variance(y_true_bin, y_prob):
    order = np.argsort(-y_prob)
    y_sorted = y_true_bin[order]
    n1 = np.sum(y_sorted == 1); n0 = np.sum(y_sorted == 0)
    if n1 == 0 or n0 == 0:
        return np.nan, np.nan
    pos_scores = y_prob[y_true_bin == 1]; neg_scores = y_prob[y_true_bin == 0]
    tx = np.array([(np.sum(neg_scores < ps) + 0.5*np.sum(neg_scores == ps)) / n0 for ps in pos_scores])
    ty = np.array([(np.sum(pos_scores > ns) + 0.5*np.sum(pos_scores == ns)) / n1 for ns in neg_scores])
    auc = np.mean(tx)
    return auc, np.var(tx, ddof=1)/n1 + np.var(ty, ddof=1)/n0

def delong_test_macro_plain(y_true, prob_a, prob_b, n_classes):
    diffs, variances = [], []
    for c in range(n_classes):
        yt_bin = (y_true == c).astype(int)
        auc_a, var_a = delong_roc_variance(yt_bin, prob_a[:, c])
        auc_b, var_b = delong_roc_variance(yt_bin, prob_b[:, c])
        if np.isnan(auc_a) or np.isnan(auc_b):
            continue
        diffs.append(auc_a - auc_b); variances.append(var_a + var_b)
    diffs = np.array(diffs); variances = np.array(variances)
    mean_diff = np.mean(diffs)
    se = np.sqrt(np.mean(variances) / len(variances))
    z = mean_diff / se
    p = 2 * (1 - stats.norm.cdf(abs(z)))
    return mean_diff, z, p

rows = []
for a, b in pairs:
    prob_a, prob_b = preds[a]['y_prob'], preds[b]['y_prob']
    auc_a = roc_auc_score(np.eye(NUM_CLASSES)[y_true], prob_a, average='macro', multi_class='ovr')
    auc_b = roc_auc_score(np.eye(NUM_CLASSES)[y_true], prob_b, average='macro', multi_class='ovr')
    mean_diff, z, p = delong_test_macro_plain(y_true, prob_a, prob_b, NUM_CLASSES)
    rows.append({'model_a': DISPLAY_NAMES[a], 'model_b': DISPLAY_NAMES[b],
                  'auc_a': auc_a, 'auc_b': auc_b, 'auc_diff': mean_diff, 'z': z, 'p_raw': p})
delong_plain_df = pd.DataFrame(rows).sort_values('p_raw').reset_index(drop=True)
m = len(delong_plain_df)
delong_plain_df['p_holm'] = (delong_plain_df['p_raw'] * (m - delong_plain_df.index)).clip(upper=1.0).cummax()
delong_plain_df['significant_holm'] = delong_plain_df['p_holm'] < 0.05
delong_plain_df.to_csv('/kaggle/working/dr_stage9_delong_plain.csv', index=False)
print("\n===== DeLong, plain =====")
print(delong_plain_df.to_string(index=False))

# ---------- 9c: DeLong, correlated (Sun & Xu 2014 fast algorithm) ----------
def compute_midrank(x):
    J = np.argsort(x); Z = x[J]; N = len(x)
    T = np.zeros(N, dtype=float)
    i = 0
    while i < N:
        j = i
        while j < N and Z[j] == Z[i]:
            j += 1
        T[i:j] = 0.5 * (i + j - 1) + 1
        i = j
    T2 = np.empty(N, dtype=float); T2[J] = T
    return T2

def fastDeLong(preds_sorted_transposed, label_1_count):
    m = label_1_count
    n = preds_sorted_transposed.shape[1] - m
    k = preds_sorted_transposed.shape[0]
    pos, neg = preds_sorted_transposed[:, :m], preds_sorted_transposed[:, m:]
    tx = np.empty([k, m]); ty = np.empty([k, n]); tz = np.empty([k, m + n])
    for r in range(k):
        tx[r, :] = compute_midrank(pos[r, :])
        ty[r, :] = compute_midrank(neg[r, :])
        tz[r, :] = compute_midrank(preds_sorted_transposed[r, :])
    aucs = tz[:, :m].sum(axis=1) / m / n - (m + 1.0) / (2.0 * n)
    v01 = (tz[:, :m] - tx) / n
    v10 = 1.0 - (tz[:, m:] - ty) / m
    sx = np.cov(v01); sy = np.cov(v10)
    delongcov = sx / m + sy / n
    return aucs, delongcov

def delong_pair_class(y_true_bin, prob_a, prob_b):
    order = np.argsort(-y_true_bin, kind='stable')
    y_sorted = y_true_bin[order]
    m = int(y_sorted.sum())
    stacked = np.vstack([prob_a[order], prob_b[order]])
    aucs, cov = fastDeLong(stacked, m)
    diff = aucs[0] - aucs[1]
    var_diff = max(cov[0,0] + cov[1,1] - 2*cov[0,1], 1e-12)
    return diff, var_diff

rows = []
for a, b in pairs:
    prob_a, prob_b = preds[a]['y_prob'], preds[b]['y_prob']
    class_diffs, class_vars = [], []
    for c in range(NUM_CLASSES):
        yt_bin = (y_true == c).astype(float)
        diff, var_diff = delong_pair_class(yt_bin, prob_a[:, c], prob_b[:, c])
        class_diffs.append(diff); class_vars.append(var_diff)
    mean_diff = np.mean(class_diffs)
    se = np.sqrt(np.sum(class_vars)) / NUM_CLASSES
    z = mean_diff / se
    p = 2 * (1 - stats.norm.cdf(abs(z)))
    rows.append({'model_a': DISPLAY_NAMES[a], 'model_b': DISPLAY_NAMES[b],
                  'auc_diff': mean_diff, 'z': z, 'p_raw': p})
delong_corr_df = pd.DataFrame(rows).sort_values('p_raw').reset_index(drop=True)
m = len(delong_corr_df)
delong_corr_df['p_holm'] = (delong_corr_df['p_raw'] * (m - delong_corr_df.index)).clip(upper=1.0).cummax()
delong_corr_df['significant_holm'] = delong_corr_df['p_holm'] < 0.05
delong_corr_df.to_csv('/kaggle/working/dr_stage9_delong_correlated.csv', index=False)
print("\n===== DeLong, CORRELATED (primary) =====")
print(delong_corr_df.to_string(index=False))

print("\nSaved: dr_stage9_mcnemar.csv, dr_stage9_delong_plain.csv, dr_stage9_delong_correlated.csv")

In [ ]:
# ===== STAGE 10: Confusion matrices, Grad-CAM, border-attention check =====
# Requires the four models still loadable and test_df from the rebuild cell.
# If starting fresh in this session, rerun the Stage 9 rebuild cell first (data + WEIGHTS_DIR + paths),
# this cell does NOT redefine test_df, aptos/eyepacs/messidor image dirs, or make_gens_test_only.

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from PIL import Image
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.applications.efficientnet import preprocess_input as eff_pre
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mob_pre
from tensorflow.keras.applications.resnet50 import preprocess_input as res_pre

GRADES = ['0','1','2','3','4']
MODEL_PATHS = {
    'Custom CNN':     (W_CUSTOM, None),
    'MobileNetV2':    (W_MOB,    mob_pre),
    'EfficientNetB0': (W_EFF,    eff_pre),
    'ResNet50':       (W_RES,    res_pre),
}
LAST_CONV_LAYER = {
    'Custom CNN': None,          # resolved dynamically below, last Conv2D by index
    'MobileNetV2': 'Conv_1',
    'EfficientNetB0': 'top_conv',
    'ResNet50': 'conv5_block3_out',
}

# ---------- 10a: CONFUSION MATRIX FIGURE, all four models, from Stage 8's saved tidy CSV ----------
cm_df = pd.read_csv('/kaggle/input/datasets/asivakumarnair/drcsvs/dr_stage8_confusion_matrices.csv')
# fallback name map: dr_stage8_confusion_matrices.csv used 'model' values like 'Custom CNN' already, confirm:
print("Models present in Stage 8 confusion matrix file:", cm_df['model'].unique())

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, model_name in zip(axes, ['Custom CNN', 'EfficientNetB0', 'MobileNetV2', 'ResNet50']):
    sub = cm_df[cm_df['model'] == model_name]
    mat = sub.pivot(index='true', columns='pred', values='count').reindex(index=range(5), columns=range(5)).values
    mat_norm = mat / mat.sum(axis=1, keepdims=True)
    im = ax.imshow(mat_norm, cmap='Blues', vmin=0, vmax=1)
    ax.set_title(model_name, fontsize=11)
    ax.set_xlabel('Predicted grade'); ax.set_ylabel('True grade')
    ax.set_xticks(range(5)); ax.set_yticks(range(5))
    for i in range(5):
        for j in range(5):
            val = mat_norm[i,j]
            ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                     color='white' if val > 0.5 else 'black', fontsize=9)
fig.suptitle('Confusion matrices, row-normalized, pooled test set (n=1,363)', fontsize=13)
fig.tight_layout()
fig.savefig('/kaggle/working/dr_stage10_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print("Saved dr_stage10_confusion_matrices.png")

# ---------- 10b: GRAD-CAM SETUP ----------
def get_last_conv_layer_name(model, model_name):
    if LAST_CONV_LAYER[model_name] is not None:
        return LAST_CONV_LAYER[model_name]
    # Custom CNN: find last Conv2D layer by index, searching the Sequential model directly
    conv_layers = [l.name for l in model.layers if isinstance(l, tf.keras.layers.Conv2D)]
    assert conv_layers, "No Conv2D layer found in Custom CNN, architecture mismatch"
    return conv_layers[-1]

def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    grad_model = tf.keras.models.Model(
        [model.inputs], [model.get_layer(last_conv_layer_name).output, model.output]
    )
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]
    grads = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy(), int(pred_index)

def load_and_preprocess_single(path, preprocess_fn, img_size=224):
    img = Image.open(path).convert('RGB').resize((img_size, img_size))
    arr = np.array(img).astype(np.float32)
    if preprocess_fn is None:
        arr = arr / 255.0
    else:
        arr = preprocess_fn(arr)
    return np.expand_dims(arr, axis=0), np.array(img)

# ---------- 10c: PICK EXEMPLAR IMAGES, one per grade, from the pooled test set ----------
exemplars = {}
for g in range(5):
    rows = test_df[test_df['grade'] == str(g)]
    if len(rows) > 0:
        exemplars[g] = rows.sample(1, random_state=SEED).iloc[0]
    else:
        print(f"WARNING: no test images for grade {g}")

print(f"\nExemplar images selected: {[(g, os.path.basename(r['image_path']), r['source']) for g, r in exemplars.items()]}")

# ---------- 10d: GENERATE GRAD-CAM GRID, all four models x five grades ----------
fig, axes = plt.subplots(4, 5, figsize=(20, 16))
border_check_notes = []

for row_idx, (model_name, (path, prep_fn)) in enumerate(MODEL_PATHS.items()):
    print(f"\nRunning Grad-CAM for {model_name}...")
    model = load_model(path)
    last_conv = get_last_conv_layer_name(model, model_name)
    print(f"  using layer: {last_conv}")

    for col_idx, g in enumerate(range(5)):
        ax = axes[row_idx, col_idx]
        if g not in exemplars:
            ax.axis('off')
            continue
        row = exemplars[g]
        img_array, orig_img = load_and_preprocess_single(row['image_path'], prep_fn)
        heatmap, pred_class = make_gradcam_heatmap(img_array, model, last_conv)

        heatmap_resized = np.array(Image.fromarray((heatmap*255).astype(np.uint8)).resize((224,224)))
        ax.imshow(orig_img)
        ax.imshow(heatmap_resized, cmap='jet', alpha=0.45)
        correct_mark = "OK" if pred_class == g else "X"
        ax.set_title(f"true={g} pred={pred_class} [{correct_mark}]\n{row['source']}", fontsize=9)
        ax.axis('off')

        # ---------- border-attention check ----------
        # flag if high-attention pixels concentrate in the outer 15% border ring of the image
        h, w = heatmap_resized.shape
        border = int(0.15 * min(h, w))
        mask = np.ones_like(heatmap_resized, dtype=bool)
        mask[border:h-border, border:w-border] = False
        border_energy = heatmap_resized[mask].astype(float).sum()
        total_energy = heatmap_resized.astype(float).sum() + 1e-8
        border_frac = border_energy / total_energy
        border_check_notes.append({
            'model': model_name, 'grade': g, 'source': row['source'],
            'border_attention_fraction': round(border_frac, 3),
            'flag': border_frac > 0.35   # threshold: >35% of attention energy in outer 15% ring
        })

    del model
    tf.keras.backend.clear_session()

fig.suptitle('Grad-CAM, one exemplar per grade, all four models\n(pooled test set, seed 42 selection)', fontsize=14)
fig.tight_layout()
fig.savefig('/kaggle/working/dr_stage10_gradcam.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print("\nSaved dr_stage10_gradcam.png")

# ---------- 10e: BORDER-ATTENTION CHECK RESULTS ----------
border_df = pd.DataFrame(border_check_notes)
border_df.to_csv('/kaggle/working/dr_stage10_border_attention.csv', index=False)
print("\n===== Border-attention check (pre-registered) =====")
print(border_df.to_string(index=False))
n_flagged = border_df['flag'].sum()
print(f"\n{n_flagged} of {len(border_df)} exemplars flagged for high border attention (>35% energy in outer 15% ring)")
if n_flagged > 0:
    print("FLAGGED CASES, review these Grad-CAM panels manually:")
    print(border_df[border_df['flag']].to_string(index=False))
    print("\nThis does NOT automatically mean shape-shortcut learning. It means these specific")
    print("panels should be visually inspected before ruling that out. Aspect ratio differs by")
    print("source (APTOS 1.322, EyePACS 1.173, Messidor 1.000), so flagged cases concentrated")
    print("on one particular source, rather than spread evenly, would be the stronger signal.")

print("\nSaved: dr_stage10_confusion_matrices.png, dr_stage10_gradcam.png, dr_stage10_border_attention.csv")

In [ ]:
# ===== STAGE 10, FULLY CORRECTED: Confusion matrices, Grad-CAM (CPU-scoped, nested-layer aware), border-attention check =====

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from PIL import Image
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.applications.efficientnet import preprocess_input as eff_pre
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mob_pre
from tensorflow.keras.applications.resnet50 import preprocess_input as res_pre

GRADES = ['0','1','2','3','4']
MODEL_PATHS = {
    'Custom CNN':     (W_CUSTOM, None),
    'MobileNetV2':    (W_MOB,    mob_pre),
    'EfficientNetB0': (W_EFF,    eff_pre),
    'ResNet50':       (W_RES,    res_pre),
}
LAST_CONV_LAYER = {
    'Custom CNN': None,          # resolved dynamically, last Conv2D by index
    'MobileNetV2': 'Conv_1',
    'EfficientNetB0': 'top_conv',
    'ResNet50': 'conv5_block3_out',
}

# ---------- 10a: CONFUSION MATRIX FIGURE, from Stage 8's locked tidy CSV ----------
cm_df = pd.read_csv('/kaggle/input/datasets/asivakumarnair/drcsvs/dr_stage8_confusion_matrices.csv')
print("Models present in Stage 8 confusion matrix file:", cm_df['model'].unique())

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, model_name in zip(axes, ['Custom CNN', 'EfficientNetB0', 'MobileNetV2', 'ResNet50']):
    sub = cm_df[cm_df['model'] == model_name]
    mat = sub.pivot(index='true', columns='pred', values='count').reindex(index=range(5), columns=range(5)).values
    mat_norm = mat / mat.sum(axis=1, keepdims=True)
    im = ax.imshow(mat_norm, cmap='Blues', vmin=0, vmax=1)
    ax.set_title(model_name, fontsize=11)
    ax.set_xlabel('Predicted grade'); ax.set_ylabel('True grade')
    ax.set_xticks(range(5)); ax.set_yticks(range(5))
    for i in range(5):
        for j in range(5):
            val = mat_norm[i,j]
            ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                     color='white' if val > 0.5 else 'black', fontsize=9)
fig.suptitle('Confusion matrices, row-normalized, pooled test set (n=1,363)', fontsize=13)
fig.tight_layout()
fig.savefig('/kaggle/working/dr_stage10_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print("Saved dr_stage10_confusion_matrices.png")

# ---------- 10b: GRAD-CAM SETUP, CPU-scoped, nested-layer aware ----------
def get_last_conv_layer_name(model, model_name):
    if LAST_CONV_LAYER[model_name] is not None:
        return LAST_CONV_LAYER[model_name]
    conv_layers = [l.name for l in model.layers if isinstance(l, tf.keras.layers.Conv2D)]
    assert conv_layers, "No Conv2D layer found in Custom CNN, architecture mismatch"
    return conv_layers[-1]

def find_layer_anywhere(model, layer_name):
    """Search top-level layers first, then descend into any nested Model/Sequential.
    build_pretrained() wraps the whole ImageNet base as one nested layer, so 'Conv_1' /
    'top_conv' / 'conv5_block3_out' live one level down, not at the top of the outer
    Sequential. Custom CNN has no nesting, so it resolves at the top-level check."""
    for layer in model.layers:
        if layer.name == layer_name:
            return layer
    for layer in model.layers:
        if isinstance(layer, tf.keras.Model):
            try:
                return layer.get_layer(layer_name)
            except ValueError:
                continue
    raise ValueError(f"Layer '{layer_name}' not found at top level or in any nested sub-model.")

def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    # CPU-scoped: FusedBatchNormGradV3 has no deterministic GPU kernel for inference-mode
    # backprop through BatchNorm, and TF_DETERMINISTIC_OPS=1 is set globally per this
    # project's locked config. Running Grad-CAM's backward pass on CPU avoids the clash
    # without disabling determinism elsewhere. Documented in the DR manual's invariant list.
    with tf.device('/CPU:0'):
        target_layer = find_layer_anywhere(model, last_conv_layer_name)
        grad_model = tf.keras.models.Model(
            [model.inputs], [target_layer.output, model.output]
        )
        img_tensor = tf.constant(img_array)
        with tf.GradientTape() as tape:
            conv_outputs, predictions = grad_model(img_tensor, training=False)
            if pred_index is None:
                pred_index = tf.argmax(predictions[0])
            class_channel = predictions[:, pred_index]
        grads = tape.gradient(class_channel, conv_outputs)
        pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
        conv_outputs = conv_outputs[0]
        heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
        heatmap = tf.squeeze(heatmap)
        heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy(), int(pred_index)

def load_and_preprocess_single(path, preprocess_fn, img_size=224):
    img = Image.open(path).convert('RGB').resize((img_size, img_size))
    arr = np.array(img).astype(np.float32)
    if preprocess_fn is None:
        arr = arr / 255.0
    else:
        arr = preprocess_fn(arr)
    return np.expand_dims(arr, axis=0), np.array(img)

# ---------- 10c: EXEMPLARS, same seed-42 selection as before ----------
exemplars = {}
for g in range(5):
    rows = test_df[test_df['grade'] == str(g)]
    if len(rows) > 0:
        exemplars[g] = rows.sample(1, random_state=SEED).iloc[0]
    else:
        print(f"WARNING: no test images for grade {g}")
print(f"\nExemplar images: {[(g, os.path.basename(r['image_path']), r['source']) for g, r in exemplars.items()]}")

# ---------- 10d: GRAD-CAM GRID, model loaded on GPU, gradient pass on CPU ----------
fig, axes = plt.subplots(4, 5, figsize=(20, 16))
border_check_notes = []

for row_idx, (model_name, (path, prep_fn)) in enumerate(MODEL_PATHS.items()):
    print(f"\nRunning Grad-CAM for {model_name}...")
    model = load_model(path)
    last_conv = get_last_conv_layer_name(model, model_name)
    print(f"  using layer: {last_conv}")

    for col_idx, g in enumerate(range(5)):
        ax = axes[row_idx, col_idx]
        if g not in exemplars:
            ax.axis('off')
            continue
        row = exemplars[g]
        img_array, orig_img = load_and_preprocess_single(row['image_path'], prep_fn)
        heatmap, pred_class = make_gradcam_heatmap(img_array, model, last_conv)

        heatmap_resized = np.array(Image.fromarray((heatmap*255).astype(np.uint8)).resize((224,224)))
        ax.imshow(orig_img)
        ax.imshow(heatmap_resized, cmap='jet', alpha=0.45)
        correct_mark = "OK" if pred_class == g else "X"
        ax.set_title(f"true={g} pred={pred_class} [{correct_mark}]\n{row['source']}", fontsize=9)
        ax.axis('off')

        h, w = heatmap_resized.shape
        border = int(0.15 * min(h, w))
        mask = np.ones_like(heatmap_resized, dtype=bool)
        mask[border:h-border, border:w-border] = False
        border_energy = heatmap_resized[mask].astype(float).sum()
        total_energy = heatmap_resized.astype(float).sum() + 1e-8
        border_frac = border_energy / total_energy
        border_check_notes.append({
            'model': model_name, 'grade': g, 'source': row['source'],
            'border_attention_fraction': round(border_frac, 3),
            'flag': border_frac > 0.35
        })

    del model
    tf.keras.backend.clear_session()

fig.suptitle('Grad-CAM, one exemplar per grade, all four models\n(pooled test set, seed 42 selection, CPU-scoped)', fontsize=14)
fig.tight_layout()
fig.savefig('/kaggle/working/dr_stage10_gradcam.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print("\nSaved dr_stage10_gradcam.png")

# ---------- 10e: BORDER-ATTENTION CHECK (pre-registered) ----------
border_df = pd.DataFrame(border_check_notes)
border_df.to_csv('/kaggle/working/dr_stage10_border_attention.csv', index=False)
print("\n===== Border-attention check =====")
print(border_df.to_string(index=False))
n_flagged = border_df['flag'].sum()
print(f"\n{n_flagged} of {len(border_df)} exemplars flagged (>35% attention energy in outer 15% ring)")
if n_flagged > 0:
    print(border_df[border_df['flag']].to_string(index=False))
    print("\nThis does not automatically mean shape-shortcut learning, it means these specific")
    print("panels warrant manual visual inspection. Flags concentrated on one source rather")
    print("than spread evenly would be the stronger signal (aspect ratio differs by source:")
    print("APTOS 1.322, EyePACS 1.173, Messidor 1.000).")

print("\nSaved: dr_stage10_confusion_matrices.png, dr_stage10_gradcam.png, dr_stage10_border_attention.csv")

In [2]:
# ===== FIX: manual two-stage Grad-CAM, sidesteps the nested-graph disconnection entirely =====
def find_layer_anywhere(model, layer_name):
    """Returns (target_layer, owning_model). owning_model is either the outer model
    itself (Custom CNN, no nesting) or the inner base model (MobileNetV2/EfficientNetB0/
    ResNet50, nested one level down by build_pretrained)."""
    for layer in model.layers:
        if layer.name == layer_name:
            return layer, model
    for layer in model.layers:
        if isinstance(layer, tf.keras.Model):
            try:
                inner = layer.get_layer(layer_name)
                return inner, layer
            except ValueError:
                continue
    raise ValueError(f"Layer '{layer_name}' not found at top level or in any nested sub-model.")

def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    with tf.device('/CPU:0'):
        target_layer, owning_model = find_layer_anywhere(model, last_conv_layer_name)
        img_tensor = tf.constant(img_array)

        if owning_model is model:
            # Custom CNN: no nesting, the original single-graph approach works fine.
            grad_model = tf.keras.models.Model([model.inputs], [target_layer.output, model.output])
            with tf.GradientTape() as tape:
                conv_outputs, predictions = grad_model(img_tensor, training=False)
                if pred_index is None:
                    pred_index = tf.argmax(predictions[0])
                class_channel = predictions[:, pred_index]
            grads = tape.gradient(class_channel, conv_outputs)

        else:
            # Pretrained models: target layer is nested inside the base model, so Keras
            # cannot trace one connected graph from the outer input to it. Instead:
            # stage 1, run the inner base model up to the target conv layer, inside the tape
            #          (this sub-model's own .input IS connected to its own .output, no nesting issue).
            # stage 2, manually apply the outer model's remaining layers (everything after
            #          the base) on top of that conv output, still inside the same tape,
            #          so gradients flow continuously from the outer prediction back
            #          through the manually-applied head layers into the conv feature map.
            feature_extractor = tf.keras.models.Model(
                owning_model.input, [target_layer.output, owning_model.output]
            )
            remaining_layers = model.layers[1:]   # everything in the outer Sequential after the base

            with tf.GradientTape() as tape:
                conv_outputs, base_output = feature_extractor(img_tensor, training=False)
                x = base_output
                for layer in remaining_layers:
                    x = layer(x, training=False)
                predictions = x
                if pred_index is None:
                    pred_index = tf.argmax(predictions[0])
                class_channel = predictions[:, pred_index]
            grads = tape.gradient(class_channel, conv_outputs)

        pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
        conv_outputs = conv_outputs[0]
        heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
        heatmap = tf.squeeze(heatmap)
        heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy(), int(pred_index)

In [1]:
# ===== FULL REBUILD, NEW SESSION: environment through Stage 10 (confusion matrices, Grad-CAM, border check) =====

!pip install -q tensorflow==2.19.0

import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'

import random
import numpy as np
import pandas as pd
import gc
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from PIL import Image
import tensorflow as tf

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print(f"Seed {SEED} set, TF {tf.__version__}, tf.keras module: {tf.keras.__name__}")

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import load_model
from tensorflow.keras.applications.efficientnet import preprocess_input as eff_pre
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mob_pre
from tensorflow.keras.applications.resnet50 import preprocess_input as res_pre
from sklearn.model_selection import train_test_split

# ---------- CONFIG ----------
APTOS_CSV     = '/kaggle/input/competitions/aptos2019-blindness-detection/train.csv'
APTOS_IMG     = '/kaggle/input/competitions/aptos2019-blindness-detection/train_images'
EYEPACS_CSV   = '/kaggle/input/datasets/benjaminwarner/resized-2015-2019-blindness-detection-images/labels/trainLabels15.csv'
EYEPACS_IMG   = '/kaggle/input/datasets/benjaminwarner/resized-2015-2019-blindness-detection-images/resized train 15'
MESSIDOR_CSV  = '/kaggle/input/datasets/mariaherrerot/messidor2preprocess/messidor_data.csv'
MESSIDOR_IMG  = '/kaggle/input/datasets/mariaherrerot/messidor2preprocess/messidor-2/messidor-2/preprocess'

WEIGHTS_DIR = '/kaggle/input/datasets/asivakumarnair/drbestmodels/'
CSV_DIR     = '/kaggle/input/datasets/asivakumarnair/drcsvs/'

GRADES      = ['0','1','2','3','4']
IMG_SIZE    = 224
BATCH_SIZE  = 32
SUBSAMPLE_SEED = 42
EYEPACS_TARGET = 3662

W_CUSTOM = WEIGHTS_DIR + 'dr_best_custom_cnn.keras'
W_EFF    = WEIGHTS_DIR + 'dr_best_efficientnet.keras'
W_MOB    = WEIGHTS_DIR + 'dr_best_mobilenet.keras'
W_RES    = WEIGHTS_DIR + 'dr_best_resnet50.keras'

for name, path in [('custom',W_CUSTOM),('eff',W_EFF),('mob',W_MOB),('res',W_RES)]:
    print(f"{name}: {'FOUND' if os.path.exists(path) else 'MISSING'} -> {path}")

# ---------- DATA REBUILD ----------
aptos = pd.read_csv(APTOS_CSV)
aptos['grade']      = aptos['diagnosis'].astype(int).astype(str)
aptos['image_path'] = APTOS_IMG + '/' + aptos['id_code'].astype(str) + '.png'
aptos['source']     = 'aptos'
aptos['patient_id'] = None

eyepacs = pd.read_csv(EYEPACS_CSV)
eyepacs['grade']      = eyepacs['level'].astype(int).astype(str)
eyepacs['image_path'] = EYEPACS_IMG + '/' + eyepacs['image'].astype(str) + '.jpg'
eyepacs['source']     = 'eyepacs'
eyepacs['patient_id'] = eyepacs['image'].str.extract(r'^(\d+)_')

messidor = pd.read_csv(MESSIDOR_CSV)
messidor['grade']      = messidor['diagnosis'].astype(int).astype(str)
messidor['image_path'] = MESSIDOR_IMG + '/' + messidor['id_code'].astype(str)
messidor['source']     = 'messidor'
messidor['patient_id'] = None

def subsample_eyepacs(df, target_n=EYEPACS_TARGET, seed=SUBSAMPLE_SEED):
    pg = df.groupby('patient_id')['grade'].max().reset_index()
    frac = target_n / len(df)
    keep, _ = train_test_split(pg, train_size=frac, stratify=pg['grade'], random_state=seed)
    return df[df['patient_id'].isin(keep['patient_id'])].reset_index(drop=True)

eyepacs_s = subsample_eyepacs(eyepacs)

def safe_split(df, label_col, test_size, rs, tag=""):
    try:
        return train_test_split(df, test_size=test_size, stratify=df[label_col], random_state=rs)
    except ValueError as e:
        print(f"WARNING [{tag}]: stratified split failed, falling back to unstratified.")
        return train_test_split(df, test_size=test_size, random_state=rs)

def split_patient_level(df, rs=SEED, tag=""):
    pg = df.groupby('patient_id')['grade'].max().reset_index()
    p_tr, p_tmp = safe_split(pg, 'grade', 0.30, rs, tag=f"{tag} first")
    p_va, p_te  = safe_split(p_tmp, 'grade', 0.50, rs, tag=f"{tag} second")
    pick = lambda ids: df[df['patient_id'].isin(ids['patient_id'])]
    return pick(p_tr), pick(p_va), pick(p_te)

def split_image_level(df, rs=SEED, tag=""):
    tr, tmp = safe_split(df, 'grade', 0.30, rs, tag=f"{tag} first")
    va, te  = safe_split(tmp, 'grade', 0.50, rs, tag=f"{tag} second")
    return tr, va, te

def split_messidor_mixed(df, rs=SEED, tag="Messidor"):
    is_im = ~df['image_path'].str.contains(r'\d{8}_\d+_\d+_PP\.png$', regex=True)
    im_df = df[is_im].copy()
    im_df['im_num'] = im_df['image_path'].str.extract(r'IM(\d+)\.JPG$').astype(int)
    im_df = im_df.sort_values('im_num').reset_index(drop=True)
    im_df['patient_id'] = 'messidor_pair_' + (im_df.index // 2).astype(str)
    pg = im_df.groupby('patient_id')['grade'].max().reset_index()
    p_tr, p_tmp = safe_split(pg, 'grade', 0.30, rs, tag=f"{tag} IM first")
    p_va, p_te  = safe_split(p_tmp, 'grade', 0.50, rs, tag=f"{tag} IM second")
    pick = lambda ids: im_df[im_df['patient_id'].isin(ids['patient_id'])]
    im_tr, im_va, im_te = pick(p_tr), pick(p_va), pick(p_te)
    date_df = df[~is_im]
    d_tr, d_va, d_te = split_image_level(date_df, rs, tag=f"{tag} date-style")
    cat = lambda a, b: pd.concat([a.drop(columns=['im_num']), b], ignore_index=True)
    return cat(im_tr, d_tr), cat(im_va, d_va), cat(im_te, d_te)

a_tr, a_va, a_te = split_image_level(aptos, tag="APTOS")
e_tr, e_va, e_te = split_patient_level(eyepacs_s, tag="EyePACS")
m_tr, m_va, m_te = split_messidor_mixed(messidor)

test_df = pd.concat([a_te, e_te, m_te], ignore_index=True)
print(f"Test set: {len(test_df):,} rows")

# ================================================================
# STAGE 10: Confusion matrices, Grad-CAM (CPU-scoped, nested-graph-safe), border-attention check
# ================================================================

MODEL_PATHS = {
    'Custom CNN':     (W_CUSTOM, None),
    'MobileNetV2':    (W_MOB,    mob_pre),
    'EfficientNetB0': (W_EFF,    eff_pre),
    'ResNet50':       (W_RES,    res_pre),
}
LAST_CONV_LAYER = {
    'Custom CNN': None,          # resolved dynamically, last Conv2D by index
    'MobileNetV2': 'Conv_1',
    'EfficientNetB0': 'top_conv',
    'ResNet50': 'conv5_block3_out',
}

# ---------- 10a: CONFUSION MATRIX FIGURE, from Stage 8's locked tidy CSV ----------
cm_df = pd.read_csv(CSV_DIR + 'dr_stage8_confusion_matrices.csv')
print("Models present in Stage 8 confusion matrix file:", cm_df['model'].unique())

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, model_name in zip(axes, ['Custom CNN', 'EfficientNetB0', 'MobileNetV2', 'ResNet50']):
    sub = cm_df[cm_df['model'] == model_name]
    mat = sub.pivot(index='true', columns='pred', values='count').reindex(index=range(5), columns=range(5)).values
    mat_norm = mat / mat.sum(axis=1, keepdims=True)
    im = ax.imshow(mat_norm, cmap='Blues', vmin=0, vmax=1)
    ax.set_title(model_name, fontsize=11)
    ax.set_xlabel('Predicted grade'); ax.set_ylabel('True grade')
    ax.set_xticks(range(5)); ax.set_yticks(range(5))
    for i in range(5):
        for j in range(5):
            val = mat_norm[i,j]
            ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                     color='white' if val > 0.5 else 'black', fontsize=9)
fig.suptitle('Confusion matrices, row-normalized, pooled test set (n=1,363)', fontsize=13)
fig.tight_layout()
fig.savefig('/kaggle/working/dr_stage10_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print("Saved dr_stage10_confusion_matrices.png")

# ---------- 10b: GRAD-CAM, CPU-scoped, handles both flat (Custom CNN) and nested (pretrained) architectures ----------
def find_layer_anywhere(model, layer_name):
    """Returns (target_layer, owning_model). owning_model is the outer model itself for
    Custom CNN (no nesting), or the inner base model for the three pretrained models,
    which build_pretrained() wraps as a single nested layer inside the outer Sequential."""
    for layer in model.layers:
        if layer.name == layer_name:
            return layer, model
    for layer in model.layers:
        if isinstance(layer, tf.keras.Model):
            try:
                inner = layer.get_layer(layer_name)
                return inner, layer
            except ValueError:
                continue
    raise ValueError(f"Layer '{layer_name}' not found at top level or in any nested sub-model.")

def get_last_conv_layer_name(model, model_name):
    if LAST_CONV_LAYER[model_name] is not None:
        return LAST_CONV_LAYER[model_name]
    conv_layers = [l.name for l in model.layers if isinstance(l, tf.keras.layers.Conv2D)]
    assert conv_layers, "No Conv2D layer found in Custom CNN, architecture mismatch"
    return conv_layers[-1]

def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    # CPU-scoped: FusedBatchNormGradV3 has no deterministic GPU kernel for inference-mode
    # backprop through BatchNorm, and TF_DETERMINISTIC_OPS=1 is set globally. Running
    # Grad-CAM's backward pass on CPU avoids the clash without disabling determinism
    # elsewhere in the pipeline.
    with tf.device('/CPU:0'):
        target_layer, owning_model = find_layer_anywhere(model, last_conv_layer_name)
        img_tensor = tf.constant(img_array)

        if owning_model is model:
            # Custom CNN: flat architecture, single connected graph, straightforward.
            grad_model = tf.keras.models.Model([model.inputs], [target_layer.output, model.output])
            with tf.GradientTape() as tape:
                conv_outputs, predictions = grad_model(img_tensor, training=False)
                if pred_index is None:
                    pred_index = tf.argmax(predictions[0])
                class_channel = predictions[:, pred_index]
            grads = tape.gradient(class_channel, conv_outputs)
        else:
            # Pretrained models: target layer is nested inside the base model. Keras cannot
            # trace one connected graph from the outer input across the nesting boundary
            # (raises "Graph disconnected"). Instead: build a sub-model from the inner
            # base's own input to the target conv layer (fully connected, no nesting issue),
            # then manually chain the outer model's remaining head layers on top, all inside
            # one GradientTape so gradients flow continuously through the manual chain.
            feature_extractor = tf.keras.models.Model(
                owning_model.input, [target_layer.output, owning_model.output]
            )
            remaining_layers = model.layers[1:]   # everything after the base in the outer Sequential
            with tf.GradientTape() as tape:
                conv_outputs, base_output = feature_extractor(img_tensor, training=False)
                x = base_output
                for layer in remaining_layers:
                    x = layer(x, training=False)
                predictions = x
                if pred_index is None:
                    pred_index = tf.argmax(predictions[0])
                class_channel = predictions[:, pred_index]
            grads = tape.gradient(class_channel, conv_outputs)

        pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
        conv_outputs = conv_outputs[0]
        heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
        heatmap = tf.squeeze(heatmap)
        heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy(), int(pred_index)

def load_and_preprocess_single(path, preprocess_fn, img_size=224):
    img = Image.open(path).convert('RGB').resize((img_size, img_size))
    arr = np.array(img).astype(np.float32)
    if preprocess_fn is None:
        arr = arr / 255.0
    else:
        arr = preprocess_fn(arr)
    return np.expand_dims(arr, axis=0), np.array(img)

# ---------- 10c: EXEMPLARS, one per grade, seed 42 ----------
exemplars = {}
for g in range(5):
    rows = test_df[test_df['grade'] == str(g)]
    if len(rows) > 0:
        exemplars[g] = rows.sample(1, random_state=SEED).iloc[0]
    else:
        print(f"WARNING: no test images for grade {g}")
print(f"\nExemplar images: {[(g, os.path.basename(r['image_path']), r['source']) for g, r in exemplars.items()]}")

# ---------- 10d: GRAD-CAM GRID ----------
fig, axes = plt.subplots(4, 5, figsize=(20, 16))
border_check_notes = []

for row_idx, (model_name, (path, prep_fn)) in enumerate(MODEL_PATHS.items()):
    print(f"\nRunning Grad-CAM for {model_name}...")
    model = load_model(path)
    last_conv = get_last_conv_layer_name(model, model_name)
    print(f"  using layer: {last_conv}")

    for col_idx, g in enumerate(range(5)):
        ax = axes[row_idx, col_idx]
        if g not in exemplars:
            ax.axis('off')
            continue
        row = exemplars[g]
        img_array, orig_img = load_and_preprocess_single(row['image_path'], prep_fn)
        heatmap, pred_class = make_gradcam_heatmap(img_array, model, last_conv)

        heatmap_resized = np.array(Image.fromarray((heatmap*255).astype(np.uint8)).resize((224,224)))
        ax.imshow(orig_img)
        ax.imshow(heatmap_resized, cmap='jet', alpha=0.45)
        correct_mark = "OK" if pred_class == g else "X"
        ax.set_title(f"true={g} pred={pred_class} [{correct_mark}]\n{row['source']}", fontsize=9)
        ax.axis('off')

        h, w = heatmap_resized.shape
        border = int(0.15 * min(h, w))
        mask = np.ones_like(heatmap_resized, dtype=bool)
        mask[border:h-border, border:w-border] = False
        border_energy = heatmap_resized[mask].astype(float).sum()
        total_energy = heatmap_resized.astype(float).sum() + 1e-8
        border_frac = border_energy / total_energy
        border_check_notes.append({
            'model': model_name, 'grade': g, 'source': row['source'],
            'border_attention_fraction': round(border_frac, 3),
            'flag': border_frac > 0.35
        })

    del model
    tf.keras.backend.clear_session()

fig.suptitle('Grad-CAM, one exemplar per grade, all four models\n(pooled test set, seed 42 selection, CPU-scoped)', fontsize=14)
fig.tight_layout()
fig.savefig('/kaggle/working/dr_stage10_gradcam.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print("\nSaved dr_stage10_gradcam.png")

# ---------- 10e: BORDER-ATTENTION CHECK (pre-registered) ----------
border_df = pd.DataFrame(border_check_notes)
border_df.to_csv('/kaggle/working/dr_stage10_border_attention.csv', index=False)
print("\n===== Border-attention check =====")
print(border_df.to_string(index=False))
n_flagged = border_df['flag'].sum()
print(f"\n{n_flagged} of {len(border_df)} exemplars flagged (>35% attention energy in outer 15% ring)")
if n_flagged > 0:
    print(border_df[border_df['flag']].to_string(index=False))
    print("\nThis does not automatically mean shape-shortcut learning, it means these specific")
    print("panels warrant manual visual inspection. Flags concentrated on one source rather")
    print("than spread evenly would be the stronger signal (aspect ratio differs by source:")
    print("APTOS 1.322, EyePACS 1.173, Messidor 1.000).")

print("\nSaved: dr_stage10_confusion_matrices.png, dr_stage10_gradcam.png, dr_stage10_border_attention.csv")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 645.0/645.0 MB 2.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 29.5 MB/s eta 0:00:00:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
ydf-tf 2.20.0 requires tensorflow==2.20.0, but you have tensorflow 2.19.0 which is incompatible.
tf-keras 2.20.0 requires tensorflow<2.21,>=2.20, but you have tensorflow 2.19.0 which is incompatible.
tensorflow-text 2.20.1 requires tensorflow<2.21,>=2.20.0, but you have tensorflow 2.19.0 which is incompatible.


2026-08-18 14:57:39.853531: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1787065059.875512      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1787065059.882660      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1787065059.900752      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1787065059.900772      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1787065059.900775      58 computation_placer.cc:177] computation placer alr

Seed 42 set, TF 2.19.0, tf.keras module: tf_keras.api._v2.keras
custom: FOUND -> /kaggle/input/datasets/asivakumarnair/drbestmodels/dr_best_custom_cnn.keras
eff: FOUND -> /kaggle/input/datasets/asivakumarnair/drbestmodels/dr_best_efficientnet.keras
mob: FOUND -> /kaggle/input/datasets/asivakumarnair/drbestmodels/dr_best_mobilenet.keras
res: FOUND -> /kaggle/input/datasets/asivakumarnair/drbestmodels/dr_best_resnet50.keras
WARNING [Messidor IM second]: stratified split failed, falling back to unstratified.
Test set: 1,363 rows
Models present in Stage 8 confusion matrix file: ['Custom CNN' 'EfficientNetB0' 'MobileNetV2' 'ResNet50']
Saved dr_stage10_confusion_matrices.png

Exemplar images: [(0, '37147_right.jpg', 'eyepacs'), (1, '2161_left.jpg', 'eyepacs'), (2, '8d7bb0649a02.png', 'aptos'), (3, '3435fd8675a2.png', 'aptos'), (4, '32851_right.jpg', 'eyepacs')]

Running Grad-CAM for Custom CNN...


I0000 00:00:1787065070.912906      58 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1787065070.915906      58 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


  using layer: conv2d_17

Running Grad-CAM for MobileNetV2...
  using layer: Conv_1

Running Grad-CAM for EfficientNetB0...
  using layer: top_conv

Running Grad-CAM for ResNet50...
  using layer: conv5_block3_out

Saved dr_stage10_gradcam.png

===== Border-attention check =====
         model  grade  source  border_attention_fraction  flag
    Custom CNN      0 eyepacs                      0.473  True
    Custom CNN      1 eyepacs                      0.434  True
    Custom CNN      2   aptos                      0.280 False
    Custom CNN      3   aptos                      0.491  True
    Custom CNN      4 eyepacs                      0.732  True
   MobileNetV2      0 eyepacs                      0.432  True
   MobileNetV2      1 eyepacs                      0.386  True
   MobileNetV2      2   aptos                      0.349 False
   MobileNetV2      3   aptos                      0.438  True
   MobileNetV2      4 eyepacs                      0.277 False
EfficientNetB0      0 eyepa

In [2]:
# ---------- 10d: GRAD-CAM GRID, WITH CLEARER LABELS AND WRONG-PREDICTION HIGHLIGHTING ----------
fig, axes = plt.subplots(4, 5, figsize=(24, 19))
border_check_notes = []

for row_idx, (model_name, (path, prep_fn)) in enumerate(MODEL_PATHS.items()):
    print(f"\nRunning Grad-CAM for {model_name}...")
    model = load_model(path)
    last_conv = get_last_conv_layer_name(model, model_name)
    print(f"  using layer: {last_conv}")

    for col_idx, g in enumerate(range(5)):
        ax = axes[row_idx, col_idx]
        if g not in exemplars:
            ax.axis('off')
            continue
        row = exemplars[g]
        img_array, orig_img = load_and_preprocess_single(row['image_path'], prep_fn)
        heatmap, pred_class = make_gradcam_heatmap(img_array, model, last_conv)

        heatmap_resized = np.array(Image.fromarray((heatmap*255).astype(np.uint8)).resize((224,224)))
        ax.imshow(orig_img)
        ax.imshow(heatmap_resized, cmap='jet', alpha=0.45)

        is_correct = (pred_class == g)

        # Bigger, bolder, color-coded title: green background patch for correct, red for wrong
        title_color = '#1a7a1a' if is_correct else '#c41111'
        verdict = 'CORRECT' if is_correct else 'WRONG'
        ax.set_title(f"TRUE={g}  PRED={pred_class}  [{verdict}]\n{model_name} | {row['source']}",
                     fontsize=12, fontweight='bold', color=title_color)

        # Thick colored border around the whole panel: green if correct, red if wrong.
        # This is the part that lets you scan the whole grid at a glance without reading text.
        border_color = '#1a7a1a' if is_correct else '#c41111'
        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_color(border_color)
            spine.set_linewidth(5)
        ax.set_xticks([]); ax.set_yticks([])

        h, w = heatmap_resized.shape
        border = int(0.15 * min(h, w))
        mask = np.ones_like(heatmap_resized, dtype=bool)
        mask[border:h-border, border:w-border] = False
        border_energy = heatmap_resized[mask].astype(float).sum()
        total_energy = heatmap_resized.astype(float).sum() + 1e-8
        border_frac = border_energy / total_energy
        border_check_notes.append({
            'model': model_name, 'grade': g, 'source': row['source'],
            'predicted': int(pred_class), 'correct': bool(is_correct),
            'border_attention_fraction': round(border_frac, 3),
            'flag': border_frac > 0.35
        })

    del model
    tf.keras.backend.clear_session()

fig.suptitle('Grad-CAM, one exemplar per grade, all four models\n'
             'Green border = correct prediction, Red border = wrong prediction\n'
             '(pooled test set, seed 42 selection, CPU-scoped)', fontsize=15, fontweight='bold')
fig.tight_layout()
fig.savefig('/kaggle/working/dr_stage10_gradcam.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print("\nSaved dr_stage10_gradcam.png")


Running Grad-CAM for Custom CNN...
  using layer: conv2d_17

Running Grad-CAM for MobileNetV2...
  using layer: Conv_1

Running Grad-CAM for EfficientNetB0...
  using layer: top_conv

Running Grad-CAM for ResNet50...
  using layer: conv5_block3_out

Saved dr_stage10_gradcam.png
